### Importing Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### Reading Minute Level Data

In [2]:
minute_2020_data = pd.read_csv('D:/PhD EMP/2020/minute_2020_data.csv').drop('Unnamed: 0', axis=1)

### Preprocessing Minute Level Dataset

In [3]:
minute_2020_data["team"] = (
    minute_2020_data["player_name"].astype(str)
    .str.split("-", n=1)
    .str[0]
)

# Ensure datetime types
minute_2020_data['date'] = pd.to_datetime(minute_2020_data['date'])
minute_2020_data['minute'] = pd.to_datetime(minute_2020_data['minute'])

# Replace date part of minute with the real date
minute_2020_data['minute'] = minute_2020_data['date'] + (minute_2020_data['minute'] - minute_2020_data['minute'].dt.normalize())

### Final Minute Level Modelling Dataset

In [4]:
minute_2020_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380193 entries, 0 to 380192
Data columns (total 66 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   player_name            380193 non-null  object        
 1   date                   380193 non-null  datetime64[ns]
 2   minute                 380193 non-null  datetime64[ns]
 3   minute_idx             380193 non-null  int64         
 4   lat_mean               380193 non-null  float64       
 5   lat_std                380193 non-null  float64       
 6   lat_min                380193 non-null  float64       
 7   lat_max                380193 non-null  float64       
 8   lon_mean               380193 non-null  float64       
 9   lon_std                380193 non-null  float64       
 10  lon_min                380193 non-null  float64       
 11  lon_max                380193 non-null  float64       
 12  speed_mean             380193 non-null  floa

### Loading Master Session 2020

In [5]:
master_session_2020 = pd.read_csv('D:/PhD EMP/2020/master_session.csv').drop("Unnamed: 0", axis=1)

master_session_2020.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3743 entries, 0 to 3742
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   player_name         3743 non-null   object 
 1   team                3743 non-null   object 
 2   session_id          3743 non-null   object 
 3   total_minutes       3743 non-null   int64  
 4   speed_mean_sess     3743 non-null   float64
 5   speed_sum_sess      3743 non-null   float64
 6   hr_mean_sess        3743 non-null   float64
 7   hr_sum_sess         3743 non-null   float64
 8   inst_acc_mean_sess  3743 non-null   float64
 9   inst_acc_sum_sess   3743 non-null   float64
 10  hacc_mean_sess      3743 non-null   float64
 11  hacc_sum_sess       3743 non-null   float64
 12  accl_x_mean_sess    3743 non-null   float64
 13  accl_x_sum_sess     3743 non-null   float64
 14  accl_y_mean_sess    3743 non-null   float64
 15  accl_y_sum_sess     3743 non-null   float64
 16  accl_z

### Creating Modelling Dataset

In [6]:
### Joning minute level and master_session
final_df = minute_2020_data.merge(
    master_session_2020[
        [
            "player_name","session_id",
            "ctl28","ctl42","daily_load","weekly_load",
            "acwr","monotony","strain",
            "sleep_duration","sleep_quality",
            "fatigue","mood","readiness","soreness","stress",
            "injury","illness"
        ]
    ],
    on=["player_name","session_id"],
    how="left"
)

### Creating Feature Groups

In [7]:
id_cols = ["player_name", "session_id", "date", "minute", "minute_idx", "team"]
target_col = "injury"

feature_cols_baseline = [
    "ctl28", "ctl42", "daily_load", "weekly_load", "acwr", "monotony", "strain",
    "sleep_duration", "sleep_quality", "fatigue", "mood", "readiness", "soreness", "stress",
    "minute_idx",
    "speed_mean", "speed_std", "speed_max",
    "heart_rate_mean", "heart_rate_std", "heart_rate_max",
    "hacc_mean", "hacc_std", "hacc_max",
    "inst_acc_impulse_mean", "inst_acc_impulse_std", "inst_acc_impulse_max",
    "accl_x_std", "accl_y_std", "accl_z_std",
    "gyro_x_std", "gyro_y_std", "gyro_z_std"
]

### Inspecting Missingness

In [8]:
missing_summary = final_df[feature_cols_baseline].isna().mean().sort_values(ascending=False)
print(missing_summary)

sleep_quality            0.497297
sleep_duration           0.497297
readiness                0.492744
fatigue                  0.492729
stress                   0.492694
soreness                 0.492542
mood                     0.492542
ctl42                    0.280108
ctl28                    0.280108
strain                   0.280108
monotony                 0.280108
acwr                     0.280108
weekly_load              0.280108
daily_load               0.280108
hacc_max                 0.000000
gyro_y_std               0.000000
gyro_x_std               0.000000
accl_z_std               0.000000
accl_y_std               0.000000
accl_x_std               0.000000
inst_acc_impulse_max     0.000000
inst_acc_impulse_std     0.000000
inst_acc_impulse_mean    0.000000
speed_std                0.000000
hacc_std                 0.000000
hacc_mean                0.000000
heart_rate_max           0.000000
heart_rate_std           0.000000
heart_rate_mean          0.000000
speed_max     

In [9]:
wellness_cols = [
    "sleep_duration","sleep_quality","fatigue",
    "mood","readiness","soreness","stress"
]

load_cols = [
    "ctl28","ctl42","daily_load","weekly_load",
    "acwr","monotony","strain"
]

for col in wellness_cols + load_cols:
    final_df[col + "_missing"] = final_df[col].isna().astype(int)

print("Missing indicators created.")

Missing indicators created.


In [10]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 380193 entries, 0 to 380192
Data columns (total 96 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   player_name             380193 non-null  object        
 1   date                    380193 non-null  datetime64[ns]
 2   minute                  380193 non-null  datetime64[ns]
 3   minute_idx              380193 non-null  int64         
 4   lat_mean                380193 non-null  float64       
 5   lat_std                 380193 non-null  float64       
 6   lat_min                 380193 non-null  float64       
 7   lat_max                 380193 non-null  float64       
 8   lon_mean                380193 non-null  float64       
 9   lon_std                 380193 non-null  float64       
 10  lon_min                 380193 non-null  float64       
 11  lon_max                 380193 non-null  float64       
 12  speed_mean              380193

### Handling Missing Values

Missing values were primarily observed in workload and wellness variables.

For each workload and wellness variable, a binary missingness indicator was
created to preserve information regarding measurement availability.

No imputation was performed at this preprocessing stage. Missing predictor
values were intentionally retained so that any required imputation can be
estimated exclusively from the training data within each modelling fold,
thereby avoiding information leakage from validation or test observations.

### Final Dataset Creation

In [11]:
target_col = ["injury"]

speed_features = [
    "speed_mean",
    "speed_std",
    "speed_min",
    "speed_max"
]

heart_features = [
    "heart_rate_mean",
    "heart_rate_std",
    "heart_rate_min",
    "heart_rate_max"
]

hacc_features = [
    "hacc_mean",
    "hacc_std",
    "hacc_min",
    "hacc_max"
]

impulse_features = [
    "inst_acc_impulse_mean",
    "inst_acc_impulse_std",
    "inst_acc_impulse_min",
    "inst_acc_impulse_max"
]

accel_features = [
    "accl_x_mean","accl_x_std","accl_x_min","accl_x_max",
    "accl_y_mean","accl_y_std","accl_y_min","accl_y_max",
    "accl_z_mean","accl_z_std","accl_z_min","accl_z_max"
]

gyro_features = [
    "gyro_x_mean","gyro_x_std","gyro_x_min","gyro_x_max",
    "gyro_y_mean","gyro_y_std","gyro_y_min","gyro_y_max",
    "gyro_z_mean","gyro_z_std","gyro_z_min","gyro_z_max"
]

load_features = [
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain"
]

wellness_features = [
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress"
]

missing_indicator_features = [
    f"{col}_missing"
    for col in load_features + wellness_features
]

# Retained as identifiers, NOT predictors
indicator_features = [
    "session_id",
    "player_name",
    "minute_idx"
]

feature_cols = (
    indicator_features +
    speed_features +
    heart_features +
    hacc_features +
    impulse_features +
    accel_features +
    gyro_features +
    load_features +
    wellness_features +
    missing_indicator_features
)

model_df = final_df[
    feature_cols + target_col
].copy()

print("Base modelling dataframe:", model_df.shape)

Base modelling dataframe: (380193, 72)


### Final Model Features Saving

In [12]:
# model_df.to_csv('D:/PhD EMP/2020/modelling_features_df.csv')

### Feature Selection Rationale

The retained variables represent pre-session workload and wellness context,
together with physiological and biomechanical measurements observed during
the session.

Their inclusion does not imply a causal or prospective relationship with
injury onset.

The subsequent analysis evaluates whether information available progressively
up to minute X can discriminate athlete-sessions occurring on days with a
recorded injury status from athlete-sessions without such a recorded status.

Variables primarily describing measurement conditions rather than athlete
state were excluded from the modelling feature set.

In [13]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 380193 entries, 0 to 380192
Data columns (total 72 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   session_id              380193 non-null  object 
 1   player_name             380193 non-null  object 
 2   minute_idx              380193 non-null  int64  
 3   speed_mean              380193 non-null  float64
 4   speed_std               380193 non-null  float64
 5   speed_min               380193 non-null  float64
 6   speed_max               380193 non-null  float64
 7   heart_rate_mean         380193 non-null  float64
 8   heart_rate_std          380193 non-null  float64
 9   heart_rate_min          380193 non-null  float64
 10  heart_rate_max          380193 non-null  float64
 11  hacc_mean               380193 non-null  float64
 12  hacc_std                380193 non-null  float64
 13  hacc_min                380193 non-null  float64
 14  hacc_max            

### Creating Data up to minute X

In [14]:
# ============================================================
# CUMULATIVE FEATURES — INFORMATION AVAILABLE UP TO MINUTE X
# ============================================================

model_df = model_df.sort_values(
    ["player_name", "session_id", "minute_idx"]
).reset_index(drop=True)

signals = [
    "speed_mean",
    "heart_rate_mean",
    "hacc_mean",
    "inst_acc_impulse_mean",
    "accl_x_std",
    "accl_y_std",
    "accl_z_std",
    "gyro_x_std",
    "gyro_y_std",
    "gyro_z_std"
]

g = model_df.groupby(
    ["player_name", "session_id"],
    sort=False
)

for col in signals:

    model_df[f"{col}_cum_mean"] = (
        g[col]
        .expanding(min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    model_df[f"{col}_cum_max"] = g[col].cummax()

    model_df[f"{col}_cum_std"] = (
        g[col]
        .expanding(min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

print("Cumulative features created.")
print(model_df.shape)

Cumulative features created.
(380193, 102)


In [15]:
# ============================================================
# DYNAMIC WITHIN-SESSION FEATURES
# ============================================================

dynamic_signals = signals

g = model_df.groupby(
    ["player_name", "session_id"],
    sort=False
)

# First differences
for col in dynamic_signals:
    model_df[f"{col}_delta"] = g[col].diff()

# Trailing rolling windows
window = 5

for col in dynamic_signals:

    model_df[f"{col}_roll_mean"] = (
        g[col]
        .rolling(window=window, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    model_df[f"{col}_roll_std"] = (
        g[col]
        .rolling(window=window, min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

# Current-minute spread features
model_df["hr_spike"] = (
    model_df["heart_rate_max"] -
    model_df["heart_rate_mean"]
)

model_df["speed_spike"] = (
    model_df["speed_max"] -
    model_df["speed_mean"]
)

model_df["impulse_spike"] = (
    model_df["inst_acc_impulse_max"] -
    model_df["inst_acc_impulse_mean"]
)

print("Dynamic features created.")
print(model_df.shape)

Dynamic features created.
(380193, 135)


In [16]:
# ============================================================
# FINAL MODELLING DATASET
# ============================================================

id_cols = [
    "session_id",
    "minute_idx",
    "player_name"
]

load_features = [
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain"
]

wellness_features = [
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress"
]

missing_indicator_features = [
    f"{col}_missing"
    for col in load_features + wellness_features
]

cumulative_features = []

for col in signals:
    cumulative_features += [
        f"{col}_cum_mean",
        f"{col}_cum_max",
        f"{col}_cum_std"
    ]

dynamic_features = []

for col in signals:
    dynamic_features += [
        f"{col}_delta",
        f"{col}_roll_mean",
        f"{col}_roll_std"
    ]

dynamic_features += [
    "hr_spike",
    "speed_spike",
    "impulse_spike"
]

target = ["injury"]

model_columns = (
    id_cols +
    load_features +
    wellness_features +
    missing_indicator_features +
    cumulative_features +
    dynamic_features +
    target
)

df_model = model_df[
    model_columns
].copy()

print("Final modelling dataset shape:", df_model.shape)

Final modelling dataset shape: (380193, 95)


In [17]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380193 entries, 0 to 380192
Data columns (total 95 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   session_id                       380193 non-null  object 
 1   minute_idx                       380193 non-null  int64  
 2   player_name                      380193 non-null  object 
 3   ctl28                            273698 non-null  float64
 4   ctl42                            273698 non-null  float64
 5   daily_load                       273698 non-null  float64
 6   weekly_load                      273698 non-null  float64
 7   acwr                             273698 non-null  float64
 8   monotony                         273698 non-null  float64
 9   strain                           273698 non-null  float64
 10  sleep_duration                   191124 non-null  float64
 11  sleep_quality                    191124 non-null  float64
 12  fa

### Temporal Feature Construction

Cumulative, first-difference, and trailing rolling-window features were
computed independently within each athlete-session after chronological
sorting.

For an observation at minute X, temporal features were constructed exclusively
from measurements available at or before minute X. No subsequent observations
from the same session were used.

Missing values arising naturally at the beginning of a session, such as the
first difference or a standard deviation based on fewer than two observations,
were intentionally retained rather than replaced by zero during preprocessing.

The injury variable represents recorded injury status on the corresponding
athlete-day. Because the available data do not provide verified within-session
injury-onset timestamps, these labels are not interpreted as confirmed injury
onset occurring after minute X.

### Writting model_df

In [18]:
df_model.to_csv(
    'D:/PhD EMP/2020/model_df.csv',
    index=False
)

In [19]:
print("Shape:", df_model.shape)

print("Sessions:", df_model["session_id"].nunique())
print("Athletes:", df_model["player_name"].nunique())

print(
    "Positive athlete-sessions:",
    df_model.loc[
        df_model["injury"] == 1,
        ["player_name", "session_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Positive athletes:",
    df_model.loc[
        df_model["injury"] == 1,
        "player_name"
    ].nunique()
)

print("\nTop missing columns:")
print(
    df_model.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("hr_trend exists:", "hr_trend" in df_model.columns)

print(
    "Duplicate athlete-session-minute rows:",
    df_model.duplicated(
        ["player_name", "session_id", "minute_idx"]
    ).sum()
)

Shape: (380193, 95)
Sessions: 251
Athletes: 48
Positive athlete-sessions: 22
Positive athletes: 5

Top missing columns:
sleep_duration              189069
sleep_quality               189069
readiness                   187338
fatigue                     187332
stress                      187319
soreness                    187261
mood                        187261
strain                      106495
ctl42                       106495
ctl28                       106495
monotony                    106495
acwr                        106495
weekly_load                 106495
daily_load                  106495
hacc_mean_cum_std             3743
speed_mean_cum_std            3743
accl_y_std_delta              3743
accl_x_std_roll_std           3743
heart_rate_mean_cum_std       3743
heart_rate_mean_roll_std      3743
dtype: int64
hr_trend exists: False
Duplicate athlete-session-minute rows: 0
